# URSim control + feedback demo
Updated: 2026-03-06

This notebook uses the setup that works in your environment:

- **RTDE receive** on **30004**
- **URScript control** on **30002**
- **Dashboard** on **29999**

Goal:
- send a command from Python,
- see the robot move in **Polyscope**,
- and see feedback (`q`, `qd`, `tcp`) in the notebook.


In [ ]:
import socket
import time
import math
from dataclasses import dataclass
from IPython.display import clear_output
import rtde_receive


In [ ]:
@dataclass
class URSimRTDEConfig:
    host: str = "127.0.0.1"
    port_rtde: int = 30004
    port_urscript: int = 30002
    port_dashboard: int = 29999

CFG = URSimRTDEConfig()
CFG


## Port checks

In [ ]:
def tcp_can_connect(host: str, port: int, timeout: float = 2.0):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True, None
    except OSError as e:
        return False, str(e)

for name, port in {
    "RTDE": CFG.port_rtde,
    "URScript": CFG.port_urscript,
    "Dashboard": CFG.port_dashboard,
}.items():
    ok, err = tcp_can_connect(CFG.host, port)
    print(f"{name:10s} ->", "OK" if ok else f"FAIL ({err})")


## Helpers

In [ ]:
def send_urscript(script: str, host=CFG.host, port=CFG.port_urscript, timeout=2.0):
    with socket.create_connection((host, port), timeout=timeout) as s:
        s.sendall(script.encode("utf-8"))

def dashboard_send(cmd: str, host=CFG.host, port=CFG.port_dashboard, timeout=2.0):
    with socket.create_connection((host, port), timeout=timeout) as s:
        banner = s.recv(4096).decode("utf-8", errors="ignore")
        s.sendall((cmd.strip() + "\n").encode("utf-8"))
        time.sleep(0.05)
        resp = s.recv(4096).decode("utf-8", errors="ignore")
    return (banner + resp).strip()

def urscript_program(lines, name="py_prog"):
    body = "\n  ".join(lines)
    return f"""def {name}():
  {body}
end
{name}()\n"""

def urscript_movej(q, a=0.3, v=0.3):
    return f"movej({list(map(float, q))}, a={a}, v={v})"

def urscript_textmsg(msg: str):
    safe = msg.replace('"', "'")
    return f'textmsg("{safe}")'


## State once

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    print("Connected:", r.isConnected())
    print("q  =", [round(v, 4) for v in r.getActualQ()])
    print("qd =", [round(v, 4) for v in r.getActualQd()])
    print("tcp=", [round(v, 4) for v in r.getActualTCPPose()])
finally:
    r.disconnect()


## Live state monitor
Stop the cell manually when done.

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    while True:
        clear_output(wait=True)
        q = r.getActualQ()
        qd = r.getActualQd()
        tcp = r.getActualTCPPose()
        print("q  =", [round(v, 4) for v in q])
        print("qd =", [round(v, 4) for v in qd])
        print("tcp=", [round(v, 4) for v in tcp])
        time.sleep(0.1)
finally:
    r.disconnect()


## Small move command test

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q0 = r.getActualQ()
finally:
    r.disconnect()

q_target = q0.copy()
q_target[0] += 0.20

prog = urscript_program([
    urscript_textmsg("move test from notebook"),
    urscript_movej(q_target, a=0.3, v=0.3),
], name="move_test")

send_urscript(prog)
print("Sent moveJ command.")
print("q0      =", [round(v, 4) for v in q0])
print("q_target=", [round(v, 4) for v in q_target])


## Command + feedback together

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q0 = r.getActualQ()
    q_target = q0.copy()
    q_target[0] += 0.20

    prog = urscript_program([
        urscript_textmsg("command + feedback demo"),
        urscript_movej(q_target, a=0.3, v=0.3),
    ], name="cmd_fb_demo")

    send_urscript(prog)

    t0 = time.time()
    while time.time() - t0 < 8.0:
        clear_output(wait=True)
        q = r.getActualQ()
        qd = r.getActualQd()
        tcp = r.getActualTCPPose()

        print("Command sent.")
        print("q0      =", [round(v, 4) for v in q0])
        print("q_target=", [round(v, 4) for v in q_target])
        print()
        print("q       =", [round(v, 4) for v in q])
        print("qd      =", [round(v, 4) for v in qd])
        print("tcp     =", [round(v, 4) for v in tcp])
        print("q_error =", [round(q_target[i] - q[i], 4) for i in range(6)])

        time.sleep(0.1)
finally:
    r.disconnect()


## Repeated trajectory demo

In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    q_center = r.getActualQ()
finally:
    r.disconnect()

q_a = q_center.copy()
q_b = q_center.copy()
q_a[0] -= 0.15
q_b[0] += 0.15

print("q_a =", [round(v, 4) for v in q_a])
print("q_b =", [round(v, 4) for v in q_b])


In [ ]:
r = rtde_receive.RTDEReceiveInterface(CFG.host)
try:
    for step, q_target in enumerate([q_a, q_b, q_a, q_b], start=1):
        prog = urscript_program([
            urscript_textmsg(f"trajectory step {step}"),
            urscript_movej(q_target, a=0.4, v=0.4),
        ], name=f"traj_{step}")
        send_urscript(prog)

        t0 = time.time()
        while time.time() - t0 < 3.0:
            clear_output(wait=True)
            q = r.getActualQ()
            tcp = r.getActualTCPPose()
            print(f"Trajectory step {step}/4")
            print("q_target =", [round(v, 4) for v in q_target])
            print("q        =", [round(v, 4) for v in q])
            print("tcp      =", [round(v, 4) for v in tcp])
            time.sleep(0.1)
finally:
    r.disconnect()


## Stop helper

In [ ]:
print(dashboard_send("stop"))


# Mujoco RTDE Loops

## Simulate one observation state
as it is received from the RTDE

In [ ]:
from URSim_RTDE_dependencies import URSimRTDEControlFeedback

robot = URSimRTDEControlFeedback()

r = robot.connect()

methods = [m for m in dir(r) if m.startswith("getActual")]
methods

In [ ]:
[m for m in dir(r) if "Torque" in m or "Current" in m]

In [7]:
import socket
import time
import math
from dataclasses import dataclass
from IPython.display import clear_output
import rtde_receive
import importlib
from URSim_RTDE_dependencies import URSimRTDEControlFeedback



robot = URSimRTDEControlFeedback()
r = robot.connect()
robot.is_connected()
try:
    for _ in range(100):
        clear_output(wait=True)
        robot.print_feedback(digits=2)
        time.sleep(0.1)
    
finally:
    robot.disconnect()
    

q      = [-1.69, -2.71, -0.31, -0.42, 1.85, 0.09]
qd     = [0.2, 0.0, 0.0, 0.0, 0.0, 0.0]
cur    = [0.0, 5.73, 1.95, 0.14, 0.01, 0.0]
tcp_xyz= [-0.21, -0.88, 0.46]
tcp_speed_3d= [0.18]
